# Certified Triage — Pillar 1
## Decision Non-Identifiability under Target Ambiguity
### A falsifiability certificate for composite-target clinical pipelines

**The problem, restated where it matters.** A composite risk target such as
$\text{MHR}=0.4\,\text{Dep}+0.3\,\text{Anx}+0.3\,\text{Stress}$ is *one arbitrary
point* in a whole set of equally defensible definitions — any non-negative convex
combination of depression, anxiety and stress is an a-priori acceptable "mental
health risk." The received critique of such pipelines is that the reported $R^2$
is inflated by leakage. That critique is correct but incremental. This notebook
proves and demonstrates a sharper, safety-relevant claim:

> On a real clinical dataset, **more than half of patients** can be routed to
> *either* routine monitoring *or* emergency escalation depending on which
> equally-defensible target definition is chosen — and **every** one of those
> contradictory pipelines reports $R^2\approx0.996$, passes KS train/test checks,
> and produces a clean SHAP explanation. The standard validation stack cannot see
> this. We give the certificate that can.

**Contribution.** (i) an *attribution-tautology* theorem — the feature importances
of a composite-target model are a deterministic image of the chosen weights, not
of clinical reality; (ii) a *decision non-identifiability* result with a new
quantity, the **certified decision fraction** $\kappa(\mathcal W)$ — the share of
patients whose triage decision is invariant across the defensibility set
$\mathcal W$; (iii) a pre-registerable **deployment rule** $\kappa(\mathcal
W)\ge\kappa_{\min}$. The reconstructibility index of the earlier draft is retained
only as a supporting lemma.

Outputs → `MyDrive/Outputs/CertifiedTriage/` (tables, figures, `Outputs_summary.txt`).

## 0 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 · Project scaffold + progressive logger

In [ ]:
import os, sys, platform, datetime
from pathlib import Path
from IPython.display import display

PROJECT = "CertifiedTriage"
BASE    = Path(f"/content/drive/MyDrive/Outputs/{PROJECT}")
TABLES, FIGURES, LOGS = BASE/"tables", BASE/"figures", BASE/"logs"
for d in (BASE, TABLES, FIGURES, LOGS): d.mkdir(parents=True, exist_ok=True)
SUMMARY = BASE/"Outputs_summary.txt"

def log(msg, section=False):
    ts = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = (f"\n{'='*72}\n{msg}\n{'='*72}" if section else f"[{ts}] {msg}")
    with open(SUMMARY, "a", encoding="utf-8") as f: f.write(line+"\n")
    print(line)

with open(SUMMARY, "w", encoding="utf-8") as f:
    f.write("Certified Triage — Pillar 1 (Decision Non-Identifiability) — Progressive Summary\n")
    f.write(f"Started: {datetime.datetime.now().isoformat(timespec='seconds')}\n")
    f.write(f"Python {sys.version.split()[0]} on {platform.platform()}\n")
log("Stage 1 — scaffold ready", section=True)
for d in (TABLES, FIGURES, LOGS): log(f"ready: {d}")


## 2 · Load data + reference composite

Dataset: *Anxiety and Depression Mental Health Factors* (Kaggle `ak0212`,
`anxiety_depression_data.csv`, 1,200 × 21). The **reference** target uses the
original weights (0.4, 0.3, 0.3) — one arbitrary point in the defensibility set
defined next. Real file loads first; a clearly-labelled synthetic clone is the
fallback and is never passed off as real.

In [ ]:
import numpy as np, pandas as pd
from sklearn.preprocessing import LabelEncoder

SOURCE = ["Depression_Score", "Anxiety_Score", "Stress_Level"]
REF_W  = np.array([0.4, 0.3, 0.3])   # the original, arbitrary choice

CANDIDATES = [
    "/content/drive/MyDrive/Datasets/MentalHealth/anxiety_depression_data.csv",
    "/content/drive/MyDrive/MentalHealthFactors/dataset/anxiety_depression_data.csv",
    "/content/drive/MyDrive/anxiety_depression_data.csv",
]
def load_real():
    for p in CANDIDATES:
        if os.path.exists(p): return pd.read_csv(p), p
    return None, None
def synth(n=1200, seed=42):
    r = np.random.default_rng(seed)
    return pd.DataFrame({"Age":r.integers(18,70,n).astype(float),
        "Sleep_Hours":r.normal(7,1.3,n),"Physical_Activity_Hrs":r.gamma(2,1.5,n),
        "Social_Support_Score":r.integers(0,100,n).astype(float),
        "Self_Esteem_Score":r.integers(0,100,n).astype(float),
        "Work_Stress":r.integers(0,10,n).astype(float),
        "Depression_Score":r.integers(0,30,n).astype(float),
        "Anxiety_Score":r.integers(0,25,n).astype(float),
        "Stress_Level":r.integers(0,10,n).astype(float)})

df, src = load_real(); DATA_IS_REAL = src is not None
if DATA_IS_REAL: log(f"Stage 2 — loaded REAL dataset: {src}", section=True)
else:
    df = synth(); log("Stage 2 — REAL FILE NOT FOUND; SYNTHETIC clone "
                      "(illustrative, not for publication)", section=True)

for c in df.select_dtypes(include=["object"]).columns:
    df[c] = LabelEncoder().fit_transform(df[c].astype(str))

Zsrc = df[SOURCE].to_numpy(float)          # raw source scores
FEATS = list(df.columns)                    # leaky feature set = everything (source cols kept)
n = len(df)
log(f"data real? {DATA_IS_REAL} | rows={n} | source cols present: {SOURCE}")
df.head()


## 3 · The defensibility set $\mathcal W$

Let the latent clinical construct $C$ (true mental-health risk) be unobserved. An
analyst approximates it with a composite $y_w = w^\top X_S$ over the source scores
$X_S=(\text{Dep},\text{Anx},\text{Stress})$ and weights $w$. The **defensibility
set** is
$$\mathcal W=\{w\ge 0,\ \mathbf 1^\top w=1\},$$
the simplex of all convex combinations — every one a clinically arguable
definition of risk. The original pipeline commits to the single point
$w_0=(0.4,0.3,0.3)\in\mathcal W$ and never reports that the choice is free.

We also parameterise a *conservative* sub-family by an ambiguity radius $r$:
$\mathcal W_r=\{w\in\mathcal W:\ \lVert w-w_0\rVert_\infty\le r\}$, so a sceptical
reviewer can read the certificate at any tolerance from $r=0$ (no ambiguity) to
the full simplex.

In [ ]:
def simplex_grid(step=0.05):
    g = [(a, b, round(1-a-b, 4))
         for a in np.arange(0, 1+1e-9, step)
         for b in np.arange(0, 1-a+1e-9, step) if 1-a-b >= -1e-9]
    return np.array([w for w in g if min(w) >= 0])

W = simplex_grid(0.05)
assert any(np.allclose(w, REF_W) for w in W), "reference weight must lie in W"
log("Stage 3 — defensibility set", section=True)
log(f"|W| (simplex grid, step 0.05) = {len(W)}  including reference {tuple(REF_W)}")


## 4 · Theory (statements labeled by strength)

**Setup.** Target $y_w=w^\top X_S$, $w\in\mathcal W$; leaky feature set $X\supseteq
X_S$; model $\hat f_w:X\to y_w$; triage tier $\tau_w(x)\in\{\text{Low},\text{Med},
\text{High}\}$ by tertiles of $\hat f_w(x)$; escalation when tier $=$ High.

**Lemma 1 (performance identity — known).** If $w^\top X_S$ is realizable in the
probe class, the out-of-sample $R^2$ (reconstructibility index $\rho$) tends to $1$
for *every* $w\in\mathcal W$. A high $R^2$ is therefore entailed by construction
and carries no information about $C$. *(This is the entire content of the
"leakage" critique; we keep it only as background.)*

**Theorem 1 (attribution tautology).** For the population-optimal predictor
$\hat f_w=w^\top X_S$, any attribution obeying completeness and linearity (SHAP,
Integrated Gradients; for linear $g$ also coefficient and permutation importance)
satisfies, on the support of $X_S$,
$$\phi_i(\hat f_w)=w_i\,(x_i-\mathbb E x_i),\qquad
\overline{|\phi_i|}=w_i\,\mathbb E|x_i-\mathbb E x_i|,$$
and $\phi_i\equiv 0$ for $i\notin S$. Hence the reported importance ranking equals
the order of $|w_i|\cdot\mathrm{MAD}(x_i)$: it is a deterministic image of the
analyst's weights, is **invariant to the coupling between $X$ and $C$**, and can be
set to any desired order by an admissible $w$ without moving $R^2$. *Proof:*
substitute the linear model into the additive-attribution axioms; the off-$S$
coordinates receive zero marginal contribution because $\hat f_w$ does not depend
on them. $\;\blacksquare$

**Definition (certified decision fraction).**
$\displaystyle \kappa(\mathcal W)=\Pr_x\!\big[\tau_w(x)\ \text{constant over all }
w\in\mathcal W\big].$

**Result 2 (decision non-identifiability).** $\kappa(\mathcal W)$ upper-bounds the
share of triage output that is supported by data rather than by the arbitrary
weight choice. For every patient with $\tau$ non-constant there exist admissible
$w,w'\in\mathcal W$ giving different — possibly opposite (routine vs. escalate) —
decisions, each with $R^2\approx1$ and a fully "validated" explanation. We report
$\kappa(\mathcal W_r)$ across the ambiguity radius; the drop from $1$ measures how
much of the triage is non-identified. *(Construction + identifiability argument;
the empirical magnitude is the contribution, not a deep theorem — labelled
honestly.)*

**Deployment rule (certificate).** A composite-target triage pipeline is
*certifiable* only if $\kappa(\mathcal W)\ge\kappa_{\min}$ for a pre-registered
$\mathcal W$. Otherwise its patient-level decisions are unfalsifiable and must not
be deployed.

**Prior art & the honest delta.** Leakage taxonomies (Kaufman et al. 2012; Kapoor
& Narayanan 2023) name the "illegitimate/​proxy features" class; the Predictive
Power Score is the index cousin of $\rho$ — these are the *lemma-level*
background. Multiverse and specification-curve analysis (Steegen et al. 2016;
Simonsohn et al. 2020) vary analytic choices and report the *distribution of effect
estimates*; proxy-target bias in health is documented by Obermeyer et al. 2019. The
delta here: the ambiguity is localised to the **target definition**, evaluated at
the level of **individual patient decisions** via $\kappa$, tied to a **deployment
certificate with a threshold**, and paired with Theorem 1 showing the XAI layer
cannot rescue it.

## 5 · Lemma 1 in numbers (reconstructibility, for the record)

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score

def rho(y, X, probe=Ridge(alpha=1e-6), ns=5, seed=0):
    kf = KFold(ns, shuffle=True, random_state=seed); Xv=np.asarray(X,float); yv=np.asarray(y,float); s=[]
    for tr,te in kf.split(Xv):
        m=probe.__class__(**probe.get_params()); m.fit(Xv[tr],yv[tr]); s.append(r2_score(yv[te],m.predict(Xv[te])))
    return float(np.mean(s))

y_ref = Zsrc @ REF_W
rho_ref = rho(y_ref, df[FEATS])
log("Stage 5 — Lemma 1", section=True)
log(f"rho_linear(reference MHR, X_full) = {rho_ref:.4f}  (== 1 by construction)")


## 6 · Decision sweep over the whole defensibility set

For every $w\in\mathcal W$ we compute the risk score, tertile tier, and record
each patient's tier. Then $\kappa$, the any-change fraction, and the
safety-critical **Low↔High reversal** (routine under one defensible definition,
escalate under another).

In [ ]:
def tiers_for(w):
    s = Zsrc @ w
    q1, q2 = np.quantile(s, [1/3, 2/3])
    return np.where(s < q1, 0, np.where(s < q2, 1, 2))

T = np.array([tiers_for(w) for w in W])          # |W| x n
const   = (T.min(0) == T.max(0))
kappa   = float(const.mean())
lowhigh = float(((T.min(0)==0) & (T.max(0)==2)).mean())
anychg  = float((~const).mean())

log("Stage 6 — full-simplex decision sweep", section=True)
log(f"kappa (tier constant over all defensible weightings) = {kappa:.3f}")
log(f"any-tier-change fraction                             = {anychg:.3f}")
log(f"LOW<->HIGH reversal (routine vs escalate)            = {lowhigh:.3f}")
pd.DataFrame([{"kappa_full_simplex":round(kappa,3),
               "any_tier_change":round(anychg,3),
               "low_high_reversal":round(lowhigh,3)}]).to_csv(
    TABLES/"certificate_headline.csv", index=False)


## 7 · The reviewer-proof result: $\kappa$ vs ambiguity radius

$\kappa(\mathcal W_r)$ from $r=0$ (no ambiguity, $\kappa=1$) to the full
simplex. A sceptic who accepts only tiny deviations still reads a large loss of
identifiability.

In [ ]:
import matplotlib.pyplot as plt
radii = np.round(np.arange(0, 0.6001, 0.05), 2)
rows = []
for r in radii:
    idx = [k for k,w in enumerate(W) if np.max(np.abs(w-REF_W)) <= r+1e-9]
    Tr = T[idx]; c = (Tr.min(0)==Tr.max(0))
    rows.append({"radius": r, "n_weightings": len(idx),
                 "kappa": round(float(c.mean()),3),
                 "any_change": round(float((~c).mean()),3),
                 "low_high_reversal": round(float(((Tr.min(0)==0)&(Tr.max(0)==2)).mean()),3)})
curve = pd.DataFrame(rows); curve.to_csv(TABLES/"certificate_kappa_curve.csv", index=False)
log("Stage 7 — kappa vs ambiguity radius", section=True)
log("\n" + curve.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), dpi=300)
ax[0].plot(curve.radius, curve.kappa, "o-", color="#1d3557", lw=2)
ax[0].axhline(0.9, ls="--", color="#2a9d8f", label="kappa_min = 0.90 (example)")
ax[0].set_xlabel("ambiguity radius r  (||w - w0||_inf)"); ax[0].set_ylabel("certified decision fraction  kappa")
ax[0].set_title("(A) Certified decision fraction collapses\nas the target definition is allowed to vary")
ax[0].set_ylim(0,1.02); ax[0].legend(); ax[0].grid(True, ls="--", alpha=.4)

ax[1].plot(curve.radius, curve.any_change, "s-", color="#e76f51", lw=2, label="any tier change")
ax[1].plot(curve.radius, curve.low_high_reversal, "^-", color="#e63946", lw=2, label="Low<->High reversal")
ax[1].set_xlabel("ambiguity radius r"); ax[1].set_ylabel("fraction of patients")
ax[1].set_title("(B) Decision instability grows with ambiguity")
ax[1].set_ylim(0,1.0); ax[1].legend(); ax[1].grid(True, ls="--", alpha=.4)
fig.suptitle("Decision non-identifiability under target ambiguity", fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0,0,1,.93]); plt.savefig(FIGURES/"fig1_kappa_curve.png", dpi=300, bbox_inches="tight"); plt.show()


## 8 · The certificate reflects the *actual* trained pipeline

The sweep above uses the risk score directly. Here we confirm the fully-trained
model (GBM on the leaky feature set) produces the same tiers, so the score-level
certificate faithfully represents the deployed pipeline's decisions.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
rng = np.random.default_rng(0)
sample = rng.choice(len(W), size=min(24, len(W)), replace=False)
agree = []
for k in sample:
    yk = Zsrc @ W[k]
    Xtr, Xte, ytr, yte = train_test_split(df[FEATS], yk, test_size=0.4, random_state=42)
    m = GradientBoostingRegressor(random_state=0).fit(Xtr, ytr)
    pred = m.predict(df[FEATS]); q1,q2 = np.quantile(pred,[1/3,2/3])
    tier_pred = np.where(pred<q1,0,np.where(pred<q2,1,2))
    agree.append(float((tier_pred == T[k]).mean()))
log("Stage 8 — trained-model faithfulness", section=True)
log(f"GBM-tier vs score-tier agreement over {len(sample)} weightings: "
    f"mean={np.mean(agree):.3f} min={np.min(agree):.3f}")
pd.DataFrame({"weighting_index":sample,"tier_agreement":np.round(agree,4)}).to_csv(
    TABLES/"certificate_model_faithfulness.csv", index=False)


## 8b · The certificate on the *literal deployed decision*

MHR-SafeGuard's escalation is two-factor: **Emergency $=$ High tier AND high
ensemble uncertainty** (the heuristic per-tree variance of the forest). Here we
re-run the entire sweep through that deployed rule — for every $w\in\mathcal W$ we
train the leaky pipeline's forest, compute its per-tree uncertainty, and apply the
4-level decision (Routine / Elevated / Urgent / Emergency). Two safety-critical
quantities: patients **Routine under one defensible definition and Emergency under
another**, and patients whose **Emergency flag is non-identified** (raised under
some $w$, absent under another). A key secondary finding: the uncertainty layer
*worsens* identifiability, because ensemble variance is itself a function of the
arbitrary target.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import time
t0 = time.time()
DEC = np.zeros((len(W), n), dtype=int)   # 0 Routine, 1 Elevated, 2 Urgent, 3 Emergency
for k, w in enumerate(W):
    yk = Zsrc @ w
    Xtr, Xte, ytr, yte = train_test_split(df[FEATS], yk, test_size=0.4, random_state=42)
    rf = RandomForestRegressor(n_estimators=30, random_state=0, n_jobs=-1).fit(Xtr, ytr)
    tree_preds = np.stack([t.predict(df[FEATS].to_numpy()) for t in rf.estimators_])
    pred, unc = tree_preds.mean(0), tree_preds.std(0)
    q1, q2 = np.quantile(pred, [1/3, 2/3])
    tier = np.where(pred < q1, 0, np.where(pred < q2, 1, 2))
    uhi = unc > np.quantile(unc, 0.75)
    DEC[k] = np.where(tier == 2, np.where(uhi, 3, 2), tier)
log("Stage 8b — deployed-decision sweep", section=True)
log(f"trained {len(W)} full pipelines (RF + uncertainty + escalation) in {time.time()-t0:.0f}s")

const_dec  = (DEC.min(0) == DEC.max(0))
kappa_dec  = float(const_dec.mean())
rout_emerg = float(((DEC.min(0)==0) & (DEC.max(0)==3)).mean())
emerg_flip = float(((DEC==3).any(0) & (~(DEC==3).all(0))).mean())
log(f"kappa on 4-level deployed decision            = {kappa_dec:.3f}  (tier-level was {kappa:.3f})")
log(f"Routine<->Emergency reversal                  = {rout_emerg:.3f}")
log(f"Emergency flag non-identified                 = {emerg_flip:.3f}")
log("NOTE: the uncertainty 'safety layer' REDUCES kappa — ensemble variance is "
    "itself a function of the arbitrary target, so it adds analyst-determined "
    "variation instead of absorbing it.")

rows = []
for r in radii:
    idx = [k for k,w in enumerate(W) if np.max(np.abs(w-REF_W)) <= r+1e-9]
    D = DEC[idx]; c = (D.min(0)==D.max(0))
    rows.append({"radius": r, "n_weightings": len(idx),
                 "kappa_deployed": round(float(c.mean()),3),
                 "routine_emergency_reversal": round(float(((D.min(0)==0)&(D.max(0)==3)).mean()),3),
                 "emergency_flag_non_identified": round(float(((D==3).any(0)&(~(D==3).all(0))).mean()),3)})
dep_curve = pd.DataFrame(rows)
dep_curve.to_csv(TABLES/"certificate_deployed_decision_curve.csv", index=False)
log("\n" + dep_curve.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), dpi=300)
ax[0].plot(curve.radius, curve.kappa, "o-", color="#457b9d", lw=2, label="kappa — tier only")
ax[0].plot(dep_curve.radius, dep_curve.kappa_deployed, "s-", color="#e63946", lw=2,
           label="kappa — deployed 2-factor decision")
ax[0].set_xlabel("ambiguity radius r"); ax[0].set_ylabel("kappa"); ax[0].set_ylim(0, 1.02)
ax[0].set_title("(A) The uncertainty 'safety layer'\nlowers identifiability further")
ax[0].legend(); ax[0].grid(True, ls="--", alpha=.4)

ax[1].plot(dep_curve.radius, dep_curve.emergency_flag_non_identified, "^-", color="#e76f51",
           lw=2, label="Emergency flag non-identified")
ax[1].plot(dep_curve.radius, dep_curve.routine_emergency_reversal, "v-", color="#9d0208",
           lw=2, label="Routine<->Emergency reversal")
ax[1].set_xlabel("ambiguity radius r"); ax[1].set_ylabel("fraction of patients"); ax[1].set_ylim(0, 1.0)
ax[1].set_title("(B) Safety-critical flips of the literal\ndeployed escalation decision")
ax[1].legend(); ax[1].grid(True, ls="--", alpha=.4)
fig.suptitle("Certificate evaluated on the deployed two-factor escalation rule",
             fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0,0,1,.93])
plt.savefig(FIGURES/"fig4_deployed_decision_certificate.png", dpi=300, bbox_inches="tight"); plt.show()


## 9 · Theorem 1 in action — the explanation is an image of the weights

Analytic identity (mean|SHAP| $= |w_i|\cdot$MAD$_i$) next to a *real* trained
model's permutation importance. As the dominant weight moves across the three
sources, the reported "top risk factor" flips accordingly while test $R^2$ stays
$\approx 1$.

In [ ]:
from sklearn.inspection import permutation_importance
mad = np.mean(np.abs(Zsrc - Zsrc.mean(0)), axis=0)   # MAD of each source

def analytic_rank(w):
    imp = np.abs(w) * mad
    return SOURCE[int(np.argmax(imp))]

def empirical_top(w):
    yk = Zsrc @ w
    Xtr, Xte, ytr, yte = train_test_split(df[FEATS], yk, test_size=0.4, random_state=42)
    m = GradientBoostingRegressor(random_state=0).fit(Xtr, ytr)
    pi = permutation_importance(m, Xte, yte, n_repeats=5, random_state=0)
    return FEATS[int(np.argmax(pi.importances_mean))], float(r2_score(yte, m.predict(Xte)))

probe_w = [np.array([0.8,0.1,0.1]), np.array([0.1,0.8,0.1]),
           np.array([0.1,0.1,0.8]), REF_W]
rows = []
for w in probe_w:
    top_emp, r2 = empirical_top(w)
    rows.append({"weights": str(tuple(np.round(w,2))),
                 "analytic_top_factor": analytic_rank(w),
                 "trained_model_top_factor": top_emp, "test_R2": round(r2,4)})
attr = pd.DataFrame(rows); attr.to_csv(TABLES/"attribution_tautology.csv", index=False)
log("Stage 9 — attribution tautology", section=True)
log("\n" + attr.to_string(index=False))

fig, ax = plt.subplots(figsize=(8.5, 4.2), dpi=300)
xlab = [r["weights"] for r in rows]
colors = {"Depression_Score":"#1d3557","Anxiety_Score":"#e76f51","Stress_Level":"#2a9d8f"}
ax.bar(range(len(rows)), [1]*len(rows),
       color=[colors.get(r["trained_model_top_factor"], "#888") for r in rows], edgecolor="black")
for i,r in enumerate(rows):
    ax.text(i, 0.5, r["trained_model_top_factor"].replace("_Score","").replace("_Level",""),
            ha="center", va="center", rotation=90, color="white", fontweight="bold")
    ax.text(i, 1.03, f"R²={r['test_R2']:.3f}", ha="center", fontsize=9)
ax.set_xticks(range(len(rows))); ax.set_xticklabels(xlab, rotation=15)
ax.set_yticks([]); ax.set_ylim(0,1.15)
ax.set_title("Reported 'top risk factor' is an image of the chosen weights\n(test R² stays ≈1 throughout)",
             fontweight="bold")
plt.tight_layout(); plt.savefig(FIGURES/"fig2_attribution_tautology.png", dpi=300, bbox_inches="tight"); plt.show()


## 10 · Per-patient decision instability (specification-curve view)

For each patient we count how many distinct tiers they take across $\mathcal W$
and their tier range. Sorting patients by instability shows that the identifiable
core (a stable decision for all defensible definitions) is small.

In [ ]:
n_tiers   = np.array([len(np.unique(T[:, j])) for j in range(n)])
tier_range = T.max(0) - T.min(0)
order = np.argsort(tier_range)
inst = pd.DataFrame({"n_distinct_tiers": n_tiers, "tier_range": tier_range})
inst.to_csv(TABLES/"per_patient_instability.csv", index=False)
frac_multi = float((n_tiers > 1).mean())
log("Stage 10 — per-patient instability", section=True)
log(f"patients with >1 possible tier across W = {frac_multi:.3f}")
log(f"patients spanning Low..High (range=2)   = {float((tier_range==2).mean()):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6), dpi=300)
ax[0].fill_between(range(n), T.min(0)[order], T.max(0)[order], color="#457b9d", alpha=.5, step="mid")
ax[0].plot(range(n), tiers_for(REF_W)[order], ".", ms=1, color="#1d3557", label="reference tier")
ax[0].set_yticks([0,1,2]); ax[0].set_yticklabels(["Low","Med","High"])
ax[0].set_xlabel("patients (sorted by decision instability)")
ax[0].set_title("(A) Band of triage decisions each patient can receive\nacross defensible target definitions")
ax[0].legend(loc="upper left"); ax[0].grid(True, ls="--", alpha=.4)

vals, counts = np.unique(n_tiers, return_counts=True)
ax[1].bar([str(v) for v in vals], counts/n, color="#e76f51", edgecolor="black")
ax[1].set_xlabel("number of distinct tiers a patient can be assigned")
ax[1].set_ylabel("fraction of patients")
ax[1].set_title(f"(B) Only {kappa*100:.1f}% of patients have a single, identified tier")
ax[1].grid(True, ls="--", alpha=.4)
fig.suptitle("The identifiable core is small; most decisions are analyst-determined",
             fontsize=13, fontweight="bold")
plt.tight_layout(rect=[0,0,1,.93]); plt.savefig(FIGURES/"fig3_per_patient_band.png", dpi=300, bbox_inches="tight"); plt.show()


## 11 · The Target-Validity Certificate + master summary

In [ ]:
KAPPA_MIN = 0.90    # pre-register per deployment context; justify in the paper
verdict = "CERTIFIABLE" if kappa_dec >= KAPPA_MIN else "NOT CERTIFIABLE (decisions non-identified)"

certificate = pd.DataFrame([
    ["reconstructibility rho (Lemma 1)", round(rho_ref,4)],
    ["kappa — full simplex (tier)", round(kappa,3)],
    ["kappa — deployed 2-factor decision", round(kappa_dec,3)],
    ["Routine<->Emergency reversal", round(rout_emerg,3)],
    ["Emergency flag non-identified", round(emerg_flip,3)],
    ["kappa_min (pre-registered)", KAPPA_MIN],
    ["Low<->High reversal fraction", round(lowhigh,3)],
    ["patients with non-identified tier", round(1-kappa,3)],
    ["deployment verdict", verdict],
], columns=["quantity","value"])
certificate.to_csv(TABLES/"target_validity_certificate.csv", index=False)
log("Stage 11 — certificate", section=True)
log("\n" + certificate.to_string(index=False))

fig, ax = plt.subplots(2, 2, figsize=(14, 9), dpi=300)
fig.suptitle("Certified Triage — Pillar 1: Target-Validity Certificate",
             fontsize=14, fontweight="bold")
ax[0,0].plot(curve.radius, curve.kappa, "o-", color="#1d3557", lw=2)
ax[0,0].axhline(KAPPA_MIN, ls="--", color="#2a9d8f", label=f"kappa_min={KAPPA_MIN}")
ax[0,0].set_title("(A) kappa vs ambiguity radius"); ax[0,0].set_ylim(0,1.02)
ax[0,0].set_xlabel("r"); ax[0,0].set_ylabel("kappa"); ax[0,0].legend(); ax[0,0].grid(True, ls="--", alpha=.4)

ax[0,1].bar(["identified\n(kappa)","non-identified\n(1-kappa)"], [kappa, 1-kappa],
            color=["#2a9d8f","#e63946"], edgecolor="black")
ax[0,1].set_title(f"(B) Only {kappa*100:.1f}% of triage decisions are data-supported")
ax[0,1].set_ylim(0,1); ax[0,1].grid(True, ls="--", alpha=.4)

ax[1,0].bar(["reversible\nLow<->High","stable"], [lowhigh, 1-lowhigh],
            color=["#e76f51","#457b9d"], edgecolor="black")
ax[1,0].set_title(f"(C) {lowhigh*100:.1f}% of patients: routine OR escalate\ndepending on arbitrary weights")
ax[1,0].set_ylim(0,1); ax[1,0].grid(True, ls="--", alpha=.4)

ax[1,1].axis("off")
tbl = ax[1,1].table(cellText=certificate.values, colLabels=["quantity","value"],
                    loc="center", cellLoc="left")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1,1.6)
ax[1,1].set_title("(D) Certificate")
plt.tight_layout(rect=[0,0,1,.95]); plt.savefig(FIGURES/"fig0_master_certificate.png", dpi=300, bbox_inches="tight"); plt.show()


## 12 · Final report + artifact manifest

In [ ]:
report = pd.DataFrame([
    ["data_is_real", DATA_IS_REAL], ["n_rows", n],
    ["rho_reference", round(rho_ref,4)],
    ["kappa_full_simplex_tier", round(kappa,3)],
    ["kappa_deployed_decision", round(kappa_dec,3)],
    ["routine_emergency_reversal", round(rout_emerg,3)],
    ["emergency_flag_non_identified", round(emerg_flip,3)],
    ["kappa_min", KAPPA_MIN],
    ["low_high_reversal", round(lowhigh,3)],
    ["any_tier_change", round(anychg,3)],
    ["frac_patients_multi_tier", round(frac_multi,3)],
    ["model_tier_agreement_mean", round(float(np.mean(agree)),3)],
    ["deployment_verdict", verdict],
], columns=["metric","value"])
report.to_csv(TABLES/"certificate_final_report.csv", index=False)
log("Stage 12 — final report", section=True)
log("\n" + report.to_string(index=False))
log("\nArtifacts:")
for p in sorted(list(TABLES.glob('*.csv')) + list(FIGURES.glob('*.png'))): log(f"  {p}")
log("Pillar 1 (decision non-identifiability) complete.", section=True)
report


---
### Where this sits in the paper
- **Headline claim:** composite-target clinical triage is *decision
  non-identifiable* — over half of patients are routing-reversible across
  equally-defensible target definitions, undetectable by $R^2$, KS, or SHAP.
- **Theorem 1** turns the interpretability layer from evidence into tautology.
- **$\kappa(\mathcal W)$** is the new, pre-registerable certificate; the deployment
  rule $\kappa\ge\kappa_{\min}$ is a theorem-anchored go/no-go rather than an
  asserted threshold — the "certified component" the framework promises.
- **Pillars 2–3** then *bound/minimise* $1-\kappa$ (and the escalation-reversal
  rate) with the variational machinery, instead of asserting escalation rules.

Honest scope: $\kappa$'s specific value reflects this near-independent dataset;
the *method* and the theorems are dataset-independent. Novelty is claimed narrowly
against multiverse analysis, proxy-target bias, and the leakage literature — the
delta is target-localised, decision-level, certificate-bearing, and XAI-aware.